# System GMM Estimation of Dual Stratification Hypothesis: Demo Notebook

## Overview
This notebook demonstrates the panel data analysis methodology for testing the dual stratification hypothesis - whether income and education inequality interactions affect democratic backsliding.

### What this notebook does:
1. Loads curated panel data on inequality and democracy indicators
2. Creates interaction terms and lagged variables for panel analysis
3. Estimates Panel OLS models with entity and time fixed effects
4. Conducts mediation analysis (inequality → political equality → democracy)
5. Evaluates hypothesis criteria for the dual stratification theory

### Key Findings (from full dataset):
- Hypothesis NOT confirmed (criteria 1 and 3 failed)
- Interaction term (gini × edu_ineq) not significant (p = 0.837)
- Mediation through political equality significant (Sobel p < 0.001)
- Triple interaction not significant (p = 0.530)

### Methods:
- Panel OLS with entity/time fixed effects
- Cluster-robust standard errors by country
- Mediation analysis using pingouin with manual Sobel-Goodman test

### Data:
- Source: V-Dem dataset (Varieties of Democracy)
- Sample: 38 countries, 1990-2023
- Complete cases: 1223 (94.7%)

In [ ]:
# Install dependencies - follows aii-colab pattern for Colab compatibility
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab (always install)
_pip('loguru')
_pip('linearmodels')
_pip('pingouin')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'statsmodels==0.14.6')

In [ ]:
# Imports - copied from original script with minimal additions for notebook
from loguru import logger
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS, PooledOLS
from linearmodels.iv import IV2SLS
from linearmodels.panel.results import PanelResults
import pingouin as pg
from typing import Dict, List, Tuple, Optional, Any
import warnings
warnings.filterwarnings('ignore')

# Additional imports for notebook visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

# Setup logging (simplified for notebook)
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
# Data loading helper - uses GitHub URL with local fallback pattern
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-27ddf1-inequality-political-equality-and-democr/main/round-2/experiment-1/demo/mini_demo_data.json"

import json, os

def load_data():
    """
    Load demo data from GitHub URL with local fallback.
    This pattern ensures compatibility with both Colab and local execution.
    """
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub URL failed: {e}")
    
    # Fallback to local file
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
# Load the demo data
data = load_data()
print(f"Data loaded successfully")
print(f"Data keys: {list(data.keys())}")
print(f"Number of examples: {len(data['data'])}")

## Configuration
Set tunable parameters for the analysis. For this demo, we use minimal values to ensure quick execution.

**Note**: To run with the full dataset, increase `n_examples` to use more data or load the complete dataset.

In [ ]:
# Configuration parameters - set to ABSOLUTE MINIMUM for demo
# Increase these gradually for more comprehensive analysis

# Data parameters
N_EXAMPLES = None  # Set to None to use all examples in mini_demo_data.json
                  # Set to a number (e.g., 50) to limit examples

# Model specification
USE_ENTITY_EFFECTS = True
USE_TIME_EFFECTS = True
COV_TYPE = 'clustered'  # 'clustered', 'robust', or 'unadjusted'

# Significance level for hypothesis testing
ALPHA = 0.05

print("Configuration set:")
print(f"  N_EXAMPLES: {N_EXAMPLES if N_EXAMPLES else 'All (mini dataset)'}")
print(f"  Entity effects: {USE_ENTITY_EFFECTS}")
print(f"  Time effects: {USE_TIME_EFFECTS}")
print(f"  Covariance type: {COV_TYPE}")
print(f"  Alpha: {ALPHA}")

## Step 1: Load and Prepare Data
Transform the raw JSON data into a panel DataFrame with proper multi-index structure (country, year).

In [ ]:
def load_and_prepare_data(data_dict: dict) -> pd.DataFrame:
    """
    Load dataset from JSON dictionary and prepare for panel analysis.
    
    Args:
        data_dict: Dictionary containing the data (from JSON)
        
    Returns:
        Prepared DataFrame with panel structure
    """
    logger.info("Loading and preparing data")
    
    # Handle different data formats
    if 'data' in data_dict:
        # iter_2 format (used in mini_demo_data.json)
        examples = data_dict['data']
        logger.info(f"Loaded {len(examples)} examples (iter_2 format)")
    elif 'datasets' in data_dict:
        # iter_1 format
        examples = data_dict['datasets'][0]['examples']
        logger.info(f"Loaded {len(examples)} examples (iter_1 format)")
    else:
        raise ValueError(f"Unknown data format. Keys: {list(data_dict.keys())}")
    
    # Convert to DataFrame
    rows = []
    for ex in examples:
        # Handle both formats
        if 'input' in ex and 'output' in ex:
            # iter_1 format
            row = json.loads(ex['input'])
            row['v2x_libdem'] = float(ex['output'])  # Dependent variable: liberal democracy index
            row['country'] = ex['metadata_country']
            row['year'] = ex['metadata_year']
            row['post_1990_democratizer'] = ex['metadata_post_1990_democratizer']
        else:
            # iter_2 format - assume keys are directly in the example
            row = ex.copy()
            if 'libdem_vdem' in row:
                row['v2x_libdem'] = row.pop('libdem_vdem')
        
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    # Ensure required columns exist
    required_cols = ['v2x_libdem', 'country', 'year']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Required column '{col}' not found in data")
    
    # Set multi-index for panel data
    df = df.set_index(['country', 'year'])
    df = df.sort_index()
    
    logger.info(f"Panel dimensions: {df.shape}")
    logger.info(f"Countries: {df.index.get_level_values('country').nunique()}")
    logger.info(f"Years: {df.index.get_level_values('year').min()} - {df.index.get_level_values('year').max()}")
    
    return df

# Load and prepare the data
df = load_and_prepare_data(data)
df.head()

## Step 2: Create Analysis Variables
Generate interaction terms, lagged variables, and within-country demeaned variables for the panel analysis.

In [ ]:
def create_variables(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create necessary variables for analysis including interactions and lags.
    
    Args:
        df: Input panel DataFrame
        
    Returns:
        DataFrame with additional variables
    """
    logger.info("Creating variables for analysis")
    
    # Reset index to access country and year as columns
    df = df.reset_index()
    
    # Create lagged dependent variable
    df['v2x_libdem_lag'] = df.groupby('country')['v2x_libdem'].shift(1)
    
    # Create interaction term: income inequality × education inequality
    df['gini_edu_interaction'] = df['gini'] * df['edu_ineq_index']
    
    # Create triple interaction: gini × edu_ineq × education_spending
    df['triple_interaction'] = df['gini_edu_interaction'] * df['education_spending_gdp']
    
    # Create lagged instruments (lags 2 and 3 for GMM)
    df['gini_lag2'] = df.groupby('country')['gini'].shift(2)
    df['gini_lag3'] = df.groupby('country')['gini'].shift(3)
    df['edu_ineq_lag2'] = df.groupby('country')['edu_ineq_index'].shift(2)
    df['edu_ineq_lag3'] = df.groupby('country')['edu_ineq_index'].shift(3)
    
    # Create within-country demeaned variables for comparison
    for col in ['gini', 'edu_ineq_index', 'gini_edu_interaction', 'education_spending_gdp', 'v2x_libdem']:
        if col in df.columns:
            country_mean = df.groupby('country')[col].transform('mean')
            df[f'{col}_within'] = df[col] - country_mean
    
    # Set index back
    df = df.set_index(['country', 'year'])
    
    logger.info(f"Created variables. DataFrame shape: {df.shape}")
    if 'gini_edu_interaction' in df.columns:
        logger.info(f"Interaction term stats: mean={df['gini_edu_interaction'].mean():.2f}, sd={df['gini_edu_interaction'].std():.2f}")
    
    return df

# Create variables
df = create_variables(df)
df.head(10)

## Step 3: Estimate Panel OLS Models
Estimate Panel OLS models with entity and time fixed effects. This is the primary estimation method (Fallback 1 from the artifact plan).

In [ ]:
def estimate_panel_ols(df: pd.DataFrame, variables: List[str], model_name: str) -> Dict[str, Any]:
    """
    Estimate Panel OLS with entity and time effects.
    
    Args:
        df: Panel DataFrame
        variables: List of variable names to include
        model_name: Name for logging
        
    Returns:
        Dictionary with estimation results
    """
    logger.info(f"Estimating {model_name} using Panel OLS")
    
    try:
        df_clean = df.dropna(subset=['v2x_libdem'] + variables)
        
        # Prepare formula
        formula = f"v2x_libdem ~ {' + '.join(variables)} + EntityEffects + TimeEffects"
        
        model = PanelOLS.from_formula(formula, data=df_clean)
        results = model.fit(cov_type=COV_TYPE)
        
        logger.info(f"{model_name} Panel OLS completed successfully")
        
        # Get number of entities correctly
        n_groups = df_clean.index.get_level_values(0).nunique() if isinstance(df_clean.index, pd.MultiIndex) else 1
        
        return {
            'model_name': model_name,
            'coefficients': {k: float(v) for k, v in results.params.to_dict().items()},
            'std_errors': {k: float(v) for k, v in results.std_errors.to_dict().items()},
            'pvalues': {k: float(v) for k, v in results.pvalues.to_dict().items()},
            'n_obs': int(results.nobs),
            'n_groups': int(n_groups),
            'r_squared': float(results.rsquared if hasattr(results, 'rsquared') else 0.0),
            'method': 'Panel OLS with entity/time effects'
        }
        
    except Exception as e:
        logger.error(f"Error estimating {model_name} with Panel OLS: {str(e)}")
        import traceback
        logger.error(traceback.format_exc())
        return {
            'model_name': model_name,
            'error': str(e)
        }

# Estimate models
models_results = {}

# Model 1: Main effect
logger.info("\n--- Model 1: Main Effect ---")
model1_vars = ['v2x_libdem_lag', 'gini', 'education_spending_gdp']
models_results['Model 1'] = estimate_panel_ols(df, model1_vars, 'Model 1: Main Effect')

# Model 2: Interaction effect
logger.info("\n--- Model 2: Interaction Effect ---")
model2_vars = ['v2x_libdem_lag', 'gini', 'edu_ineq_index', 'gini_edu_interaction', 'education_spending_gdp']
models_results['Model 2'] = estimate_panel_ols(df, model2_vars, 'Model 2: Interaction Effect')

# Model 4: Triple interaction
logger.info("\n--- Model 4: Triple Interaction ---")
model4_vars = ['v2x_libdem_lag', 'gini', 'edu_ineq_index', 'gini_edu_interaction', 
               'triple_interaction', 'education_spending_gdp']
models_results['Model 4'] = estimate_panel_ols(df, model4_vars, 'Model 4: Triple Interaction')

print("\nPanel OLS estimation complete!")

## Step 4: Mediation Analysis
Test whether political equality (v2pepwrsoc) mediates the relationship between inequality interaction and democracy.

In [ ]:
def mediation_analysis(df: pd.DataFrame, x: str, m: str, y: str) -> Dict[str, Any]:
    """
    Perform mediation analysis using pingouin with manual Sobel test fallback.
    
    Args:
        df: DataFrame (not multi-index)
        x: Independent variable (inequality interaction)
        m: Mediator (political equality)
        y: Dependent variable (democracy index)
        
    Returns:
        Dictionary with mediation results
    """
    logger.info(f"Performing mediation analysis: {x} -> {m} -> {y}")
    
    try:
        # Use pingouin for mediation
        med = pg.mediation_analysis(data=df, x=x, m=m, y=y, seed=42, n_boot=N_BOOT)
        
        # Extract Sobel test results
        sobel_row = med[med['path'] == 'Indirect']
        if len(sobel_row) > 0:
            sobel_p = sobel_row['pval'].values[0]
        else:
            sobel_p = None
        
        return {
            'x': x,
            'm': m,
            'y': y,
            'sobel_p': float(sobel_p) if sobel_p is not None else None,
            'n': len(df),
            'paths': med.to_dict('records')
        }
        
    except Exception as e:
        logger.warning(f"Pingouin mediation failed: {e}. Using manual Sobel test.")
        return manual_sobel_test(df, x, m, y)

def manual_sobel_test(df: pd.DataFrame, x: str, m: str, y: str) -> Dict[str, Any]:
    """
    Manual Sobel-Goodman test for mediation.
    
    Args:
        df: DataFrame
        x: Independent variable
        m: Mediator
        y: Dependent variable
        
    Returns:
        Dictionary with Sobel test results
    """
    logger.info("Running manual Sobel-Goodman test")
    
    # Path a: X -> M
    model_a = sm.OLS(df[m], sm.add_constant(df[x])).fit()
    a_coef = model_a.params[x]
    a_se = model_a.bse[x]
    
    # Path b: M -> Y (controlling for X)
    model_b = sm.OLS(df[y], sm.add_constant(df[[x, m]])).fit()
    b_coef = model_b.params[m]
    b_se = model_b.bse[m]
    
    # Sobel test
    sobel_z = (a_coef * b_coef) / np.sqrt((a_coef**2 * b_se**2) + (b_coef**2 * a_se**2))
    sobel_p = 2 * (1 - stats.norm.cdf(abs(sobel_z)))
    
    return {
        'x': x,
        'm': m,
        'y': y,
        'sobel_z': float(sobel_z),
        'sobel_p': float(sobel_p),
        'n': len(df),
        'paths': [
            {'path': f'{m} ~ X', 'coef': float(a_coef), 'se': float(a_se), 'pval': float(model_a.pvalues[x])},
            {'path': f'Y ~ {m}', 'coef': float(b_coef), 'se': float(b_se), 'pval': float(model_b.pvalues[m])}
        ]
    }

# Configuration for mediation
N_BOOT = 100  # Number of bootstrap samples (set low for demo speed)

# Run mediation analysis
df_reset = df.reset_index()
mediation_result = mediation_analysis(df_reset, 'gini_edu_interaction', 'v2pepwrsoc', 'v2x_libdem')
models_results['Model 3_mediation'] = mediation_result

print("\nMediation analysis complete!")

## Step 5: Hypothesis Test Evaluation
Evaluate whether the dual stratification hypothesis is confirmed based on three criteria.

In [ ]:
# Evaluate hypothesis criteria
criterion1 = False  # interaction negative and significant
criterion2 = False  # mediation significant
criterion3 = False  # triple interaction positive and significant

# Check criterion 1: interaction term in Model 2
if 'Model 2' in models_results and 'coefficients' in models_results['Model 2']:
    coef = models_results['Model 2']['coefficients'].get('gini_edu_interaction', None)
    pval = models_results['Model 2']['pvalues'].get('gini_edu_interaction', None)
    if coef is not None and pval is not None:
        criterion1 = (coef < 0) and (pval < ALPHA)
        print(f"Criterion 1 (interaction negative and significant): {criterion1}")
        print(f"  Coefficient: {coef:.6f}, p-value: {pval:.4f}")

# Check criterion 2: mediation
if 'Model 3_mediation' in models_results and 'sobel_p' in models_results['Model 3_mediation']:
    sobel_p = models_results['Model 3_mediation']['sobel_p']
    criterion2 = (sobel_p is not None) and (sobel_p < ALPHA)
    print(f"\nCriterion 2 (mediation significant): {criterion2}")
    print(f"  Sobel p-value: {sobel_p:.6f}")

# Check criterion 3: triple interaction in Model 4
if 'Model 4' in models_results and 'coefficients' in models_results['Model 4']:
    coef = models_results['Model 4']['coefficients'].get('triple_interaction', None)
    pval = models_results['Model 4']['pvalues'].get('triple_interaction', None)
    if coef is not None and pval is not None:
        criterion3 = (coef > 0) and (pval < ALPHA)
        print(f"\nCriterion 3 (triple interaction positive and significant): {criterion3}")
        print(f"  Coefficient: {coef:.6f}, p-value: {pval:.4f}")

hypothesis_confirmed = criterion1 and criterion2 and criterion3
print(f"\n{'='*60}")
print(f"HYPOTHESIS CONFIRMED: {hypothesis_confirmed}")
print(f"{'='*60}")

## Results Visualization
Display key results in tables and create visualizations to understand the relationships.

In [ ]:
# Visualize results - coefficient plot
def plot_coefficients(models_results):
    """
    Create a coefficient plot for all models.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('Panel OLS Coefficient Estimates with 95% Confidence Intervals', fontsize=14)
    
    model_names = ['Model 1', 'Model 2', 'Model 4']
    colors = ['steelblue', 'coral', 'mediumseagreen']
    
    for idx, (model_name, color) in enumerate(zip(model_names, colors)):
        if model_name not in models_results or 'coefficients' not in models_results[model_name]:
            continue
        
        coefs = models_results[model_name]['coefficients']
        errors = models_results[model_name]['std_errors']
        
        # Filter out EntityEffects and TimeEffects
        keys = [k for k in coefs.keys() if k not in ['EntityEffects', 'TimeEffects']]
        
        # Prepare data for plotting
        coef_values = [coefs[k] for k in keys]
        ci = 1.96 * np.array([errors[k] for k in keys])
        
        # Horizontal plot
        ax = axes[idx]
        y_pos = np.arange(len(keys))
        
        ax.errorbar(coef_values, y_pos, xerr=ci, fmt='o', color=color, capsize=5)
        ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(keys, fontsize=9)
        ax.set_xlabel('Coefficient Estimate')
        ax.set_title(model_name)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Plot coefficients
plot_coefficients(models_results)

In [ ]:
# Print summary table of results
print("="*80)
print("SUMMARY OF PANEL OLS RESULTS")
print("="*80)

for model_name in ['Model 1', 'Model 2', 'Model 4']:
    if model_name not in models_results:
        print(f"\n{model_name}: Not estimated")
        continue

    result = models_results[model_name]

    if 'error' in result:
        print(f"\n{model_name}: Error - {result['error']}")
        continue

    print(f"\n{model_name}: {result.get('model_name', model_name)}")
    print("-"*60)
    print(f"Observations: {result.get('n_obs', 'N/A')}")
    print(f"Countries: {result.get('n_groups', 'N/A')}")
    print(f"R-squared: {result.get('r_squared', 'N/A')}")
    print("\nCoefficients:")
    print(f"  {'Variable':<30} {'Coef':>12} {'Std Err':>12} {'p-value':>10}")
    print(f"  {'-'*30} {'-'*12} {'-'*12} {'-'*10}")

    if 'coefficients' in result:
        for var in result['coefficients'].keys():
            if var in ['EntityEffects', 'TimeEffects']:
                continue
            coef = result['coefficients'].get(var, float('nan'))
            se = result['std_errors'].get(var, float('nan')) if 'std_errors' in result else float('nan')
            pval = result['pvalues'].get(var, float('nan')) if 'pvalues' in result else float('nan')
            sig = '*' if not np.isnan(pval) and pval < 0.05 else ''
            print(f"  {var:<30} {coef:>12.6f} {se:>12.6f} {pval:>10.4f}{sig}")

print("\n" + "="*80)
print("MEDIATION ANALYSIS RESULTS")
print("="*80)
if 'Model 3_mediation' in models_results:
    med = models_results['Model 3_mediation']
    print(f"\nMediator: {med.get('m', 'N/A')}")
    print(f"Sobel p-value: {med.get('sobel_p', 'N/A')}")
    print(f"\nPaths:")
    for path in med.get('paths', []):
        print(f"  {path.get('path', 'N/A'):<25} coef={path.get('coef', float('nan')):>10.6f}, se={path.get('se', float('nan')):>10.6f}, p={path.get('pval', float('nan')):>10.6f}")

print("\n" + "="*80)
print("HYPOTHESIS TEST RESULTS")
print("="*80)
print(f"Criterion 1 (interaction negative and significant): {criterion1}")
print(f"Criterion 2 (mediation significant): {criterion2}")
print(f"Criterion 3 (triple interaction positive and significant): {criterion3}")
print(f"\nHYPOTHESIS CONFIRMED: {hypothesis_confirmed}")

In [ ]:
# Visualize the relationship between inequality and democracy
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter plot: Gini vs Democracy
ax1 = axes[0]
df_reset = df.reset_index()
ax1.scatter(df_reset['gini'], df_reset['v2x_libdem'], alpha=0.6, color='steelblue')
ax1.set_xlabel('Gini Coefficient (Income Inequality)')
ax1.set_ylabel('V-Dem Liberal Democracy Index')
ax1.set_title('Income Inequality vs. Democracy')
ax1.grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(df_reset['gini'], df_reset['v2x_libdem'], 1)
p = np.poly1d(z)
ax1.plot(df_reset['gini'], p(df_reset['gini']), "r--", alpha=0.8)

# Scatter plot: Interaction term vs Democracy
ax2 = axes[1]
ax2.scatter(df_reset['gini_edu_interaction'], df_reset['v2x_libdem'], alpha=0.6, color='coral')
ax2.set_xlabel('Gini × Education Inequality Interaction')
ax2.set_ylabel('V-Dem Liberal Democracy Index')
ax2.set_title('Inequality Interaction vs. Democracy')
ax2.grid(True, alpha=0.3)

# Add trend line
z2 = np.polyfit(df_reset['gini_edu_interaction'], df_reset['v2x_libdem'], 1)
p2 = np.poly1d(z2)
ax2.plot(df_reset['gini_edu_interaction'], p2(df_reset['gini_edu_interaction']), "r--", alpha=0.8)

plt.tight_layout()
plt.show()